### 사전학습된 seq2seq 모델을 활용해 번역

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = 'facebook/nllb-200-distilled-600M'  # 다국어 번역 hugging-face 오픈소스 모델

# 영어 -> 한국어 번역을 위한 토크나이저
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang='eng_Latn', tgt_lang='kor_Hang')
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)  # 사전학습된 Seq2Seq2 모델 로드
model.eval()  # 추론 모드

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

c:\Users\Playdata\NLP\nlp_venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--facebook--nllb-200-distilled-600M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=Tr

In [2]:
# 추론 함수 : 영어 문장 리스트 -> 한국어 번역
def translate_en_to_ko(texts):
    inputs = tokenizer(
        texts,                # 번역할 문장 리스트
        return_tensors='pt',  # Pytorch Tensor 형태로 변환
        padding=True          # 문장 길이 맞추는 padding
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,    # 토큰화된 입력값
            # 생성 문장 시작 언어를 한국어로 지정
            forced_bos_token_id = tokenizer.convert_tokens_to_ids("kor_Hang"),
            max_new_tokens = 50  # 생성할 최대 토큰 수
        )

    translations = tokenizer.batch_decode(
        outputs,  # 모델이 생성한 토큰 ID
        skip_special_tokens=True  # 특수 토큰 제거
    )

    return translations  # 번역 결과 리스트 반환

In [3]:
texts = [
    "I love studying artificial intelligence.",
    "She is learning Python Programming",
    "The weather is beautiful today"
]

translations = translate_en_to_ko(texts)

for eng, kor in zip(texts, translations):
    print(f"영어 원문 : {eng}")
    print(f"번역문 : {kor}\n")

[transformers] Both `max_new_tokens` (=50) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


영어 원문 : I love studying artificial intelligence.
번역문 : 저는 인공지능을 공부하는 것을 좋아합니다.

영어 원문 : She is learning Python Programming
번역문 : 그녀는 파이썬 프로그래밍을 배우고 있습니다.

영어 원문 : The weather is beautiful today
번역문 : 날씨가 아름답습니다.

